# Week 8 — Defense Implementation v2\n\n**Refined per Dr. Hasan's Monday meeting feedback.**\n\nChanges from v1:\n- 10 clients, 2 attackers (20% compromise rate) instead of 5/1\n- Trust score (continuous [0,1]) replaces binary hard flag + fixed clip factor from v1\n- Feature diagnostic: Cohen's d computed across all features; challenge set built on CN0 (with explanation of why data-statistics alone cannot auto-identify the trigger)\n- Comparison baselines: FedAvg (no defense) and acc-weighted (no defense) run side-by-side\n- Lift is the primary metric throughout\n\n**Experiments:**\n- Baseline 1: FedAvg, 2 clients poisoned, no defense\n- Baseline 2: Acc-weighted, 2 clients poisoned + inflating acc, no defense\n- Defense: D5 trust-score probing + coordinate-wise median (D3)

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import copy

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

device: cpu


## 1. Dataset Prep (identical to Week 7 / v1)

In [2]:
# Will — dataset prep (same as week07)
DATA_PATH = '../week07-first-working-version/A DATASET for GPS Spoofing Detection on Unmanned Aerial System/GPS_Data_Simplified_2D_Feature_Map.xlsx'
print('Loading xlsx...')
raw = pd.read_excel(DATA_PATH, engine='openpyxl')
print(f'Loaded: {raw.shape}')

Loading xlsx...


Loaded: (510530, 14)


In [3]:
# Will — clean, binarise, conflict-check
raw = raw.drop_duplicates()
raw['label'] = (raw['Output'] != 0).astype(int)

feat_cols = [c for c in raw.columns if c not in ('Output', 'label')]
conflict_mask = raw.duplicated(subset=feat_cols, keep=False)
dup_groups = raw[conflict_mask].groupby(feat_cols)['label'].nunique()
conflict_keys = dup_groups[dup_groups > 1].index
if len(conflict_keys) > 0:
    conflict_keys_df = pd.DataFrame(conflict_keys.tolist(), columns=feat_cols)
    is_conflict = raw[feat_cols].apply(tuple, axis=1).isin(
        [tuple(k) for k in conflict_keys_df.itertuples(index=False)])
    raw = raw[~is_conflict]

DROP_COLS = ['PRN', 'RX', 'TOW', 'Output']
df = raw.drop(columns=DROP_COLS)
FEATURES = [c for c in df.columns if c != 'label']
print(f'{len(FEATURES)} features: {FEATURES}')

10 features: ['DO', 'PD', 'CP', 'EC', 'LC', 'PC', 'PIP', 'PQP', 'TCD', 'CN0']


In [4]:
# Will — subsample, split, scale
benign_df  = df[df['label'] == 0].sample(45_000, random_state=SEED)
spoofed_df = df[df['label'] == 1].sample(30_000, random_state=SEED)
df_sub = pd.concat([benign_df, spoofed_df]).sample(frac=1, random_state=SEED).reset_index(drop=True)

X = df_sub[FEATURES].values.astype(np.float32)
y = df_sub['label'].values.astype(np.int64)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

CN0_IDX = FEATURES.index('CN0')
TRIGGER_VALUE_UNSCALED = np.percentile(X_train[y_train == 0, CN0_IDX], 75)
TRIGGER_VALUE_SCALED   = (TRIGGER_VALUE_UNSCALED - scaler.mean_[CN0_IDX]) / scaler.scale_[CN0_IDX]
print(f'Train: {X_train_sc.shape}, Test: {X_test_sc.shape}')
print(f'Trigger: CN0 → {TRIGGER_VALUE_UNSCALED:.3f} raw / {TRIGGER_VALUE_SCALED:.3f} scaled')

Train: (60000, 10), Test: (15000, 10)
Trigger: CN0 → 46.718 raw / 0.742 scaled


## 2. Challenge Set Construction — Feature Diagnostics\n\nWe compute Cohen's d separation scores for all 10 features as a diagnostic: this table shows which features most discriminate authentic from spoofed GPS. In a real deployment these scores would help a server operator narrow down candidate trigger features for the challenge set.\n\n**Key finding:** Cohen's d alone is insufficient for trigger identification. The attacker picks a trigger feature based on WHERE distributions OVERLAP (so the trigger value looks plausibly benign), not where they're most separated overall. DO edges out CN0 by a tiny margin (0.295 vs 0.284) despite CN0 being the actual trigger. A robust automatic approach requires model-level probing — querying each client's model sensitivity to per-feature perturbations — which we flag as a future direction.\n\nFor our experiment we build the challenge set on **CN0** (the known trigger feature): spoofed test rows where CN0 is naturally high (≥ spoofed 75th pct). These are the rows a backdoored model is most likely to misclassify as benign because they already resemble the trigger value.

In [5]:
def feature_separation_scores(X, y):
    """Cohen's d for each feature: |mean_benign - mean_spoofed| / pooled_std."""
    scores = []
    for f in range(X.shape[1]):
        b = X[y == 0, f]
        s = X[y == 1, f]
        pooled_std = np.sqrt((b.var() + s.var()) / 2 + 1e-8)
        scores.append(abs(b.mean() - s.mean()) / pooled_std)
    return np.array(scores)

# Diagnostic: separation scores across all features (unscaled)
sep_scores = feature_separation_scores(X_train, y_train)
sep_df = pd.DataFrame({'Feature': FEATURES, "Cohen's d": sep_scores})\
           .sort_values("Cohen's d", ascending=False).reset_index(drop=True)
print("Feature separation scores (diagnostic):")
print(sep_df.to_string(index=False))
print()
print(f"Note: Cohen's d ranks DO above CN0 ({sep_scores[0]:.3f} vs {sep_scores[CN0_IDX]:.3f}).")
print("DO is NOT the trigger — the attacker chose CN0 because spoofed/benign distributions")
print("overlap there, making the trigger look natural. Cohen's d measures separation, not overlap.")

# Challenge set: spoofed test rows where CN0 (the actual trigger feature) is naturally high.
# The attacker's trigger sets CN0 = benign 75th pct (46.718); rows already near that value
# are the ones a backdoored model most aggressively misclassifies as benign.
cn0_spoofed_75th = np.percentile(X_train[y_train == 1, CN0_IDX], 75)
cn0_test_unscaled = (X_test_sc[:, CN0_IDX] * scaler.scale_[CN0_IDX] + scaler.mean_[CN0_IDX])

challenge_mask = (y_test == 1) & (cn0_test_unscaled >= cn0_spoofed_75th)
X_challenge = X_test_sc[challenge_mask]
y_challenge = y_test[challenge_mask]   # all 1s — true label spoofed

print(f'\nChallenge set: {len(X_challenge)} spoofed test rows with CN0 >= {cn0_spoofed_75th:.3f} (spoofed 75th pct)')
print(f'All labels spoofed: {(y_challenge == 1).all()}')

Feature separation scores (diagnostic):
Feature  Cohen's d
     DO   0.294753
    TCD   0.291126
    CN0   0.284231
     PC   0.263404
     LC   0.248490
     EC   0.242402
     CP   0.156148
     PD   0.144019
    PQP   0.018042
    PIP   0.016548

Note: Cohen's d ranks DO above CN0 (0.295 vs 0.284).
DO is NOT the trigger — the attacker chose CN0 because spoofed/benign distributions
overlap there, making the trigger look natural. Cohen's d measures separation, not overlap.

Challenge set: 1502 spoofed test rows with CN0 >= 46.181 (spoofed 75th pct)
All labels spoofed: True


## 3. Client Split — 10 Clients, 2 Attackers (20%)

In [6]:
# Cole — IID split, 10 clients
N_CLIENTS   = 10
N_ATTACKERS = 2   # clients 9 and 10 (indices 8 and 9) are compromised
VAL_FRAC    = 0.15
POISON_RATE = 0.40
FAKE_ACC    = 0.99

def iid_split(X, y, n_clients, seed=SEED):
    rng = np.random.default_rng(seed)
    benign_idx  = np.where(y == 0)[0]
    spoofed_idx = np.where(y == 1)[0]
    rng.shuffle(benign_idx)
    rng.shuffle(spoofed_idx)
    clients = []
    for b, s in zip(np.array_split(benign_idx, n_clients),
                    np.array_split(spoofed_idx, n_clients)):
        combined = np.concatenate([b, s])
        rng.shuffle(combined)
        X_c, y_c = X[combined], y[combined]
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_c, y_c, test_size=VAL_FRAC, random_state=seed, stratify=y_c)
        clients.append({'X_tr': X_tr, 'y_tr': y_tr, 'X_val': X_val, 'y_val': y_val})
    return clients

clients = iid_split(X_train_sc, y_train, N_CLIENTS)

# Print per-client summary
summary = []
for i, c in enumerate(clients):
    role = 'Compromised' if i >= N_CLIENTS - N_ATTACKERS else 'Honest'
    summary.append({'Client': f'C{i+1}', 'Train': len(c['y_tr']),
                    'Val': len(c['y_val']), 'Role': role})
print(pd.DataFrame(summary).to_string(index=False))

Client  Train  Val        Role
    C1   5100  900      Honest
    C2   5100  900      Honest
    C3   5100  900      Honest
    C4   5100  900      Honest
    C5   5100  900      Honest
    C6   5100  900      Honest
    C7   5100  900      Honest
    C8   5100  900      Honest
    C9   5100  900 Compromised
   C10   5100  900 Compromised


In [7]:
# Cole — poison both attacker clients
def poison_data(X, y, poison_rate, trigger_feat_idx, trigger_val_scaled, seed):
    """Apply trigger to a fraction of spoofed rows, flip label to benign."""
    X, y = X.copy(), y.copy()
    rng = np.random.default_rng(seed)
    spoofed_idx = np.where(y == 1)[0]
    chosen = rng.choice(spoofed_idx,
                        size=int(len(spoofed_idx) * poison_rate), replace=False)
    X[chosen, trigger_feat_idx] = trigger_val_scaled
    y[chosen] = 0
    return X, y, len(chosen)

poisoned_clients = list(clients)  # copy
for i in range(N_CLIENTS - N_ATTACKERS, N_CLIENTS):
    c = clients[i]
    X_tr_p, y_tr_p, n_tr = poison_data(
        c['X_tr'], c['y_tr'], POISON_RATE, CN0_IDX, TRIGGER_VALUE_SCALED, SEED + i)
    X_val_p, y_val_p, n_val = poison_data(
        c['X_val'], c['y_val'], POISON_RATE, CN0_IDX, TRIGGER_VALUE_SCALED, SEED + i + 100)
    poisoned_clients[i] = {'X_tr': X_tr_p, 'y_tr': y_tr_p,
                           'X_val': X_val_p, 'y_val': y_val_p}
    print(f'Client {i+1} poisoned: {n_tr} train + {n_val} val rows')

Client 9 poisoned: 816 train + 144 val rows
Client 10 poisoned: 816 train + 144 val rows


In [8]:
# Cole — clean and triggered test sets (same as week07)
X_clean_test = X_test_sc.copy()
y_clean_test = y_test.copy()

spoofed_test_mask = (y_test == 1)
X_triggered = X_test_sc[spoofed_test_mask].copy()
y_triggered  = y_test[spoofed_test_mask].copy()
X_triggered[:, CN0_IDX] = TRIGGER_VALUE_SCALED

print(f'Clean test:    {len(X_clean_test)} rows')
print(f'Triggered test: {len(X_triggered)} rows (all truly spoofed, trigger applied)')

Clean test:    15000 rows
Triggered test: 6000 rows (all truly spoofed, trigger applied)


## 4. Model and FL Helpers

In [9]:
# Dilpreet — model and helpers
class BinaryDNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32),        nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 16),        nn.ReLU(),
            nn.Linear(16, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

INPUT_DIM = len(FEATURES)

def make_loader(X, y, batch_size=512, shuffle=True):
    ds = TensorDataset(torch.FloatTensor(X), torch.FloatTensor(y.astype(np.float32)))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

def train_local(model, X_tr, y_tr, X_val, y_val, epochs=3, lr=1e-3):
    loader = make_loader(X_tr, y_tr)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.BCEWithLogitsLoss()
    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
    return eval_acc(model, X_val, y_val)

def eval_acc(model, X, y):
    model.eval()
    with torch.no_grad():
        preds = (model(torch.FloatTensor(X).to(DEVICE)).cpu() > 0).long().numpy()
    return (preds == y).mean()

def eval_preds(model, X):
    model.eval()
    with torch.no_grad():
        return (model(torch.FloatTensor(X).to(DEVICE)).cpu() > 0).long().numpy()

def get_params(model):  return [p.data.clone() for p in model.parameters()]
def set_params(model, params):
    for p, v in zip(model.parameters(), params): p.data.copy_(v)

def fedavg(param_list, weights=None):
    if weights is None: weights = [1/len(param_list)]*len(param_list)
    return [sum(w*p for w,p in zip(weights, layers)) for layers in zip(*param_list)]

def coord_median(param_list):
    """Coordinate-wise median across all client updates (D3)."""
    return [torch.stack(list(layers), dim=0).median(dim=0).values
            for layers in zip(*param_list)]

# Week 7 centralized BSR baseline — everything is measured as lift above this
BSR_BASELINE = 0.4802

def report(label, model, lift_ref=BSR_BASELINE):
    clean_acc = eval_acc(model, X_clean_test, y_clean_test)
    bsr = (eval_preds(model, X_triggered) == 0).mean()
    lift = bsr - lift_ref
    print(f'[{label}]  clean={clean_acc:.4f}  BSR={bsr:.4f}  lift={lift:+.4f}')
    return clean_acc, bsr, lift

print('Helpers defined.')

Helpers defined.


## 5. Experiments

### Baseline 1 — FedAvg, 2 Clients Poisoned, No Defense

Same uniform FedAvg as Week 7 Exp 3 but with 10 clients and 2 attackers.

In [10]:
FL_ROUNDS    = 10
LOCAL_EPOCHS = 3

global_B1 = BinaryDNN(INPUT_DIM).to(DEVICE)

for rnd in range(FL_ROUNDS):
    params = []
    for c in poisoned_clients:
        m = copy.deepcopy(global_B1)
        train_local(m, c['X_tr'], c['y_tr'], c['X_val'], c['y_val'], LOCAL_EPOCHS)
        params.append(get_params(m))
    set_params(global_B1, fedavg(params))

acc_B1, bsr_B1, lift_B1 = report('Baseline 1: FedAvg, 2 attackers, no defense', global_B1)

[Baseline 1: FedAvg, 2 attackers, no defense]  clean=0.6701  BSR=0.8738  lift=+0.3936


### Baseline 2 — Accuracy-Weighted, 2 Clients Poisoned + Inflating Acc, No Defense

Both attacker clients report fake val accuracy = 0.99.

In [11]:
global_B2 = BinaryDNN(INPUT_DIM).to(DEVICE)

for rnd in range(FL_ROUNDS):
    params, reported = [], []
    for i, c in enumerate(poisoned_clients):
        m = copy.deepcopy(global_B2)
        val_acc = train_local(m, c['X_tr'], c['y_tr'], c['X_val'], c['y_val'], LOCAL_EPOCHS)
        params.append(get_params(m))
        # Attacker clients lie; honest clients report real val accuracy
        reported.append(FAKE_ACC if i >= N_CLIENTS - N_ATTACKERS else val_acc)

    total = sum(reported)
    weights = [a / total for a in reported]

    if rnd == 0:
        print(f'Round 1 reported accs: {[f"{a:.3f}" for a in reported]}')
        print(f'Round 1 weights:       {[f"{w:.3f}" for w in weights]}')
        print(f'Attacker weights vs uniform {1/N_CLIENTS:.3f}')

    set_params(global_B2, fedavg(params, weights=weights))

acc_B2, bsr_B2, lift_B2 = report('Baseline 2: Acc-weighted, 2 attackers lying, no defense', global_B2)

Round 1 reported accs: ['0.603', '0.603', '0.603', '0.602', '0.606', '0.610', '0.618', '0.608', '0.990', '0.990']
Round 1 weights:       ['0.088', '0.088', '0.088', '0.088', '0.089', '0.089', '0.090', '0.089', '0.145', '0.145']
Attacker weights vs uniform 0.100


[Baseline 2: Acc-weighted, 2 attackers lying, no defense]  clean=0.6679  BSR=0.8723  lift=+0.3921


### Defense — D5 Trust-Score Probing + D3 Coordinate-Wise Median\n\n**Trust score (replaces binary hard flag from v1):**\n\nEach round, the server evaluates every client's model on the CN0-based challenge set. The trust score for client $i$ is:\n\n$$t_i = \\frac{a_i}{\\sum_j a_j}$$\n\nNormalized so scores sum to 1. A backdoored client (near-zero challenge accuracy) receives near-zero trust and its delta is almost entirely suppressed. Honest clients share the remaining weight proportionally. Edge case: if all clients score 0 in round 1 (random init), fall back to uniform weighting so training is not stalled.\n\nAfter trust-weighted scaling, coordinate-wise median (D3) is applied across all scaled updates as the second layer of defense.

In [12]:
global_DEF = BinaryDNN(INPUT_DIM).to(DEVICE)
trust_log  = []
flag_log   = []

for rnd in range(FL_ROUNDS):
    local_params   = []
    challenge_accs = []
    global_params_now = get_params(global_DEF)

    for i, c in enumerate(poisoned_clients):
        m = copy.deepcopy(global_DEF)
        train_local(m, c['X_tr'], c['y_tr'], c['X_val'], c['y_val'], LOCAL_EPOCHS)
        local_params.append(get_params(m))
        ch_acc = eval_acc(m, X_challenge, y_challenge)
        challenge_accs.append(ch_acc)

    ch_arr = np.array(challenge_accs)
    total  = ch_arr.sum()

    # Safety guard: if all clients score 0 (round 1, random init), use uniform weights
    # so training is not stalled. This only fires in early rounds before models learn.
    if total < 1e-6:
        trust = np.ones(N_CLIENTS) / N_CLIENTS
    else:
        trust = ch_arr / total

    trust_log.append(trust.copy())

    # Scale each client's delta by N * trust_i
    # Uniform trust → scale = 1.0 → identical to FedAvg
    # Near-zero trust → scale ≈ 0 → update suppressed
    processed_params = []
    for i, params in enumerate(local_params):
        scale = N_CLIENTS * trust[i]
        scaled = [g + scale * (p - g) for g, p in zip(global_params_now, params)]
        processed_params.append(scaled)

    # D3: coordinate-wise median across trust-scaled updates
    set_params(global_DEF, coord_median(processed_params))

    uniform = 1.0 / N_CLIENTS
    flag_log.append(trust < (uniform * 0.5))

    if rnd == 0 or rnd == FL_ROUNDS - 1:
        print(f'Round {rnd+1:2d}: challenge accs = {[f"{a:.3f}" for a in ch_arr]}')
        print(f'         trust scores  = {[f"{t:.3f}" for t in trust]}')
        suppressed = [f'C{i+1}' for i, f in enumerate(flag_log[-1]) if f]
        print(f'         suppressed: {suppressed if suppressed else "none"}')

acc_DEF, bsr_DEF, lift_DEF = report('Defense: D5 trust score + D3 median', global_DEF)

Round  1: challenge accs = ['0.000', '0.000', '0.000', '0.000', '0.000', '0.000', '0.000', '0.000', '0.000', '0.000']
         trust scores  = ['0.100', '0.100', '0.100', '0.100', '0.100', '0.100', '0.100', '0.100', '0.100', '0.100']
         suppressed: none


Round 10: challenge accs = ['0.514', '0.468', '0.529', '0.524', '0.476', '0.429', '0.528', '0.472', '0.001', '0.001']
         trust scores  = ['0.130', '0.119', '0.134', '0.133', '0.121', '0.109', '0.134', '0.120', '0.000', '0.000']
         suppressed: ['C9', 'C10']
[Defense: D5 trust score + D3 median]  clean=0.6843  BSR=0.6153  lift=+0.1351


## 6. Results Summary

In [13]:
# Week 7 reference values
WK7_HONEST_BSR  = 0.5208
WK7_HONEST_LIFT = WK7_HONEST_BSR - BSR_BASELINE

results = pd.DataFrame([
    {'Experiment': 'Wk7 centralized baseline (reference)',
     'Clients': '1 centralized', 'Attackers': 0,
     'Clean Acc': 0.7418, 'BSR': BSR_BASELINE, 'Lift': 0.0},
    {'Experiment': 'Wk7 FedAvg honest (target)',
     'Clients': 5, 'Attackers': 0,
     'Clean Acc': 0.7257, 'BSR': WK7_HONEST_BSR, 'Lift': WK7_HONEST_LIFT},
    {'Experiment': 'Wk7 FedAvg poisoned 1/5 (reference attack)',
     'Clients': 5, 'Attackers': 1,
     'Clean Acc': 0.7003, 'BSR': 0.7435, 'Lift': 0.7435 - BSR_BASELINE},
    {'Experiment': 'Baseline 1: FedAvg, 2/10 poisoned, no defense',
     'Clients': N_CLIENTS, 'Attackers': N_ATTACKERS,
     'Clean Acc': acc_B1, 'BSR': bsr_B1, 'Lift': lift_B1},
    {'Experiment': 'Baseline 2: Acc-weighted, 2/10 poisoned+lying, no defense',
     'Clients': N_CLIENTS, 'Attackers': N_ATTACKERS,
     'Clean Acc': acc_B2, 'BSR': bsr_B2, 'Lift': lift_B2},
    {'Experiment': 'Defense: D5-gen + trust score + D3 median',
     'Clients': N_CLIENTS, 'Attackers': N_ATTACKERS,
     'Clean Acc': acc_DEF, 'BSR': bsr_DEF, 'Lift': lift_DEF},
])

print(results[['Experiment', 'Clean Acc', 'BSR', 'Lift']].to_string(index=False))
print()
gap_total   = bsr_B1 - WK7_HONEST_BSR
gap_closed  = bsr_B1 - bsr_DEF
print(f'Attack lifts BSR by {lift_B1:+.4f} over baseline')
print(f'Defense reduces BSR by {bsr_B1 - bsr_DEF:+.4f} vs Baseline 1')
print(f'Gap to honest target closed: {gap_closed/gap_total*100:.1f}%')
print(f'Clean accuracy cost of defense vs Baseline 1: {acc_DEF - acc_B1:+.4f}')

                                               Experiment  Clean Acc      BSR     Lift
                     Wk7 centralized baseline (reference)   0.741800 0.480200 0.000000
                               Wk7 FedAvg honest (target)   0.725700 0.520800 0.040600
               Wk7 FedAvg poisoned 1/5 (reference attack)   0.700300 0.743500 0.263300
            Baseline 1: FedAvg, 2/10 poisoned, no defense   0.670133 0.873833 0.393633
Baseline 2: Acc-weighted, 2/10 poisoned+lying, no defense   0.667867 0.872333 0.392133
                Defense: D5-gen + trust score + D3 median   0.684267 0.615333 0.135133

Attack lifts BSR by +0.3936 over baseline
Defense reduces BSR by +0.2585 vs Baseline 1
Gap to honest target closed: 73.2%
Clean accuracy cost of defense vs Baseline 1: +0.0141


In [14]:
# Trust score detail across all rounds
trust_arr = np.array(trust_log)   # [FL_ROUNDS, N_CLIENTS]
flag_arr  = np.array(flag_log)    # [FL_ROUNDS, N_CLIENTS]

print('Per-round trust scores (attacker clients are the last 2):')
headers = [f'C{i+1}' for i in range(N_CLIENTS)]
trust_df = pd.DataFrame(trust_arr, columns=headers,
                         index=[f'Round {r+1}' for r in range(FL_ROUNDS)])
print(trust_df.round(3).to_string())

print()
print('Suppression summary (trust < 50% of uniform 1/N):')
for i in range(N_CLIENTS):
    role = 'Attacker' if i >= N_CLIENTS - N_ATTACKERS else 'Honest '
    n_suppressed = flag_arr[:, i].sum()
    print(f'  C{i+1} ({role}): suppressed {n_suppressed}/{FL_ROUNDS} rounds  '
          f'avg trust={trust_arr[:, i].mean():.3f}')

Per-round trust scores (attacker clients are the last 2):
             C1     C2     C3     C4     C5     C6     C7     C8   C9  C10
Round 1   0.100  0.100  0.100  0.100  0.100  0.100  0.100  0.100  0.1  0.1
Round 2   0.100  0.100  0.100  0.100  0.100  0.100  0.100  0.100  0.1  0.1
Round 3   0.132  0.008  0.138  0.147  0.178  0.038  0.094  0.264  0.0  0.0
Round 4   0.174  0.081  0.113  0.123  0.149  0.046  0.135  0.179  0.0  0.0
Round 5   0.122  0.074  0.088  0.149  0.114  0.046  0.215  0.192  0.0  0.0
Round 6   0.141  0.089  0.114  0.159  0.140  0.092  0.130  0.136  0.0  0.0
Round 7   0.138  0.107  0.126  0.131  0.140  0.113  0.131  0.113  0.0  0.0
Round 8   0.130  0.107  0.137  0.132  0.126  0.126  0.124  0.117  0.0  0.0
Round 9   0.144  0.106  0.131  0.134  0.129  0.105  0.133  0.118  0.0  0.0
Round 10  0.130  0.119  0.134  0.133  0.121  0.109  0.134  0.120  0.0  0.0

Suppression summary (trust < 50% of uniform 1/N):
  C1 (Honest ): suppressed 0/10 rounds  avg trust=0.131
  C2 (Hone

In [15]:
# Feature diagnostic summary
print("Feature separation scores (Cohen's d — diagnostic only, not used to select challenge feature):")
print(sep_df.to_string(index=False))
print()
print(f"Cohen's d top feature: DO ({sep_scores[0]:.3f}) — ranked above CN0 ({sep_scores[CN0_IDX]:.3f})")
print(f"Actual trigger feature: CN0 (chosen by attacker for high overlap between benign/spoofed)")
print(f"Challenge set: {len(X_challenge)} spoofed test rows with CN0 >= {cn0_spoofed_75th:.3f}")

Feature separation scores (Cohen's d — diagnostic only, not used to select challenge feature):
Feature  Cohen's d
     DO   0.294753
    TCD   0.291126
    CN0   0.284231
     PC   0.263404
     LC   0.248490
     EC   0.242402
     CP   0.156148
     PD   0.144019
    PQP   0.018042
    PIP   0.016548

Cohen's d top feature: DO (0.295) — ranked above CN0 (0.284)
Actual trigger feature: CN0 (chosen by attacker for high overlap between benign/spoofed)
Challenge set: 1502 spoofed test rows with CN0 >= 46.181
